In [1]:
import pandas as pd
import numpy as np

import polars as pl

from tqdm import tqdm
import requests

In [2]:
data = pl.read_csv("new_pullreq.csv").to_pandas()
data.head()

,id,project_id,github_id,pull_request_id,ownername,reponame,merged_or_not,lifetime_minutes,mergetime_minutes,num_commits,...,doc_files_open,other_files_open,src_churn_open,test_churn_open,code_churn_open,churn_addition_open,churn_deletion_open,code_chunk_num_open,commits_on_files_touched_close,test_inclusion_open
0,1,13708387,1895,16946031,stylelint,stylelint,1,237,237.0,1,...,0,1,0,0,0,0,0,1,79,0
1,2,100297899,353,51228565,Joaogarciadelima,checklistos,0,1410,NaN,1,...,0,0,0,0,0,0,0,1,38,0
2,3,93139005,404,42975776,binary-com,SmartCharts,1,4,4.0,1,...,0,1,0,0,0,0,0,1,175,0
3,4,15059440,3434,34700062,letsencrypt,boulder,1,52,52.0,1,...,0,0,9,104,113,110,3,2,24,1
4,5,29684214,486,34175163,PyCQA,astroid,1,2414,2414.0,1,...,0,0,33,27,60,60,0,2,7,1


In [3]:
print(data.shape)

(3281386, 140)


In [3]:
print(data[data["merged_or_not"] == 1].shape)

(2765736, 140)


In [4]:
import requests

def get_added_lines(diff):
    return '\n'.join(line[1:] for line in diff.split('\n') if line.startswith('+'))

def get_pull_request_files(owner, repo, pull_number, github_token):
    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pull_number}/files"
    headers = {
        'Accept': 'application/vnd.github+json',
        'Authorization': f'Bearer {github_token}',
        'X-GitHub-Api-Version': '2022-11-28',
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()

def get_added_code(owner, repo, pull_number, github_token):
    files = get_pull_request_files(owner, repo, pull_number, github_token)
    for file in files:
        return get_added_lines(file['patch'])

In [5]:
github_token = 'ENTER GITHUB TOKEN HERE'
invalid_repos = []

In [6]:
def get_code(row_id):
    owner = data.iloc[row_id]["ownername"]
    reponame = data.iloc[row_id]["reponame"]
    github_id = data.iloc[row_id]["github_id"]
    try: 
        code = get_added_code(owner, reponame, github_id, github_token)
        data.loc[row_id, "added_code"] = code
    except:
        invalid_repos.append(github_id)

In [ ]:
import concurrent.futures

i = 0
max_val = data.shape[0]
with concurrent.futures.ThreadPoolExecutor() as e:
    fut = [e.submit(get_code, i) for i in range(0,  max_val)]
    for r in concurrent.futures.as_completed(fut):
            print(f"Finished {i}")
            i+=1

In [ ]:
data.to_csv("pullreq_with_code.csv")